# SEMMA: Student Performance Prediction

This notebook applies the SAS-oriented **SEMMA** methodology to the uploaded Kaggle/UCI Student Performance data. The five stages are **Sample, Explore, Modify, Model, and Assess**.

**Primary objective:** predict final mathematics grade and identify students who may need academic support.

**Important target caution:** `G1` and `G2` are previous grades and strongly predict `G3`. They are appropriate for an end-of-term prediction model, but they should not be used to claim that the model provides an early intervention before those grades are available.

In [ ]:
%pip -q install pandas numpy scikit-learn matplotlib seaborn joblib

In [ ]:
from pathlib import Path
import json, warnings, joblib
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score, roc_auc_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_curve)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
warnings.filterwarnings("ignore"); sns.set_theme(style="whitegrid"); RANDOM_STATE=42

## 1. SEMMA — Sample

The primary sample is the mathematics dataset. We use a reproducible stratified train/test split for the early-warning classification task and a reproducible train/test split for the final-grade regression task. The Portuguese dataset and the provided `student-merge.R` script are retained as an optional cross-subject extension.

In [ ]:
filename="student-mat.csv"
candidates=[Path(filename),Path("/content")/filename,Path("upload")/filename,Path("/workspace/scratch/16abedd5f8ac/upload")/filename,Path("projects/semma/artifacts/student-mat.csv")]
data_path=next((p for p in candidates if p.exists()),None)
if data_path is None:
    try:
        from google.colab import files
        uploaded=files.upload(); data_path=Path(next(iter(uploaded)))
    except Exception as exc:
        raise FileNotFoundError("Upload student-mat.csv and rerun this cell") from exc
df=pd.read_csv(data_path,sep=";")
print("Using:",data_path,"Shape:",df.shape)
display(df.head())

In [ ]:
sample_info=pd.DataFrame({"dtype":df.dtypes.astype(str),"missing":df.isna().sum(),"unique":df.nunique()})
display(sample_info)
print("Duplicate rows:",df.duplicated().sum())
print("Grade sample size:",len(df))
X_base=df.drop(columns=["G3"]); y_grade=df["G3"].astype(float)
risk=(df["G3"]<10).astype(int)
X_train_g,X_test_g,y_train_g,y_test_g=train_test_split(X_base,y_grade,test_size=.20,random_state=RANDOM_STATE)
X_train_r,X_test_r,y_train_r,y_test_r=train_test_split(X_base,risk,test_size=.20,stratify=risk,random_state=RANDOM_STATE)
print("Regression sample:",X_train_g.shape,X_test_g.shape)
print("Risk sample:",X_train_r.shape,X_test_r.shape)

## 2. SEMMA — Explore

Explore distributions, academic outcomes, absences, study time, and demographic/social variables. Exploration is descriptive and does not justify causal claims.

In [ ]:
display(df.describe(include="all").T)
fig,axes=plt.subplots(1,3,figsize=(16,4))
sns.histplot(df["G3"],bins=21,discrete=True,ax=axes[0],color="steelblue")
sns.boxplot(data=df,x="studytime",y="G3",ax=axes[1])
sns.scatterplot(data=df,x="absences",y="G3",hue="school",ax=axes[2])
axes[0].set_title("Final grade G3"); axes[1].set_title("G3 by study time"); axes[2].set_title("G3 and absences")
plt.tight_layout(); plt.show()

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(13,4))
sns.boxplot(data=df,x="failures",y="G3",ax=axes[0]); axes[0].set_title("G3 by prior failures")
sns.countplot(data=df,x="school",hue="sex",ax=axes[1]); axes[1].set_title("Sample by school and sex")
plt.tight_layout(); plt.show()
print("At-risk rate G3 < 10:",round(risk.mean()*100,2),"%")
display(df.groupby("school")[["G1","G2","G3","absences"]].mean().round(2))

In [ ]:
numeric=df.select_dtypes(include=["number"])
plt.figure(figsize=(11,7)); sns.heatmap(numeric.corr(),cmap="coolwarm",center=0); plt.title("Numeric correlation matrix"); plt.show()
print("G1/G2/G3 correlation:"); display(df[["G1","G2","G3"]].corr().round(3))

## 3. SEMMA — Modify

Two feature views are created.

- **End-of-term grade model:** includes `G1` and `G2`, because the objective is to predict final grade after prior grading information is available.
- **Early-warning risk model:** excludes `G1` and `G2`, because using them would make an early-intervention claim misleading. The risk label is `G3 < 10`.

Categorical variables are one-hot encoded. Numerical variables are imputed and standardized inside each train-fitted pipeline.

In [ ]:
def make_preprocessor(X):
    nums=X.select_dtypes(include=["number"]).columns.tolist(); cats=X.select_dtypes(exclude=["number"]).columns.tolist()
    return ColumnTransformer([
        ("num",Pipeline([("impute",SimpleImputer(strategy="median")),("scale",StandardScaler())]),nums),
        ("cat",Pipeline([("impute",SimpleImputer(strategy="most_frequent")),("encode",OneHotEncoder(handle_unknown="ignore"))]),cats),
    ])
X_grade=df.drop(columns=["G3"])
X_early=X_grade.drop(columns=["G1","G2"])
y_grade=df["G3"].astype(float); y_risk=(df["G3"]<10).astype(int)
Xg_train,Xg_test,yg_train,yg_test=train_test_split(X_grade,y_grade,test_size=.20,random_state=RANDOM_STATE)
Xr_train,Xr_test,yr_train,yr_test=train_test_split(X_early,y_risk,test_size=.20,stratify=y_risk,random_state=RANDOM_STATE)
print("Grade features:",X_grade.shape[1],"Early-warning features:",X_early.shape[1])

## 4. SEMMA — Model

The grade task uses Linear Regression as a transparent baseline and Random Forest Regression as a nonlinear benchmark. The early-warning task uses Logistic Regression and Decision Tree classification. Model choices are evaluated in the next SEMMA stage.

In [ ]:
grade_models={"linear_regression":Pipeline([("pre",make_preprocessor(Xg_train)),("model",LinearRegression())]),"random_forest_regressor":Pipeline([("pre",make_preprocessor(Xg_train)),("model",RandomForestRegressor(n_estimators=300,min_samples_leaf=3,random_state=RANDOM_STATE,n_jobs=-1))])}
risk_models={"logistic_regression":Pipeline([("pre",make_preprocessor(Xr_train)),("model",LogisticRegression(max_iter=1000,class_weight="balanced",random_state=RANDOM_STATE))]),"decision_tree":Pipeline([("pre",make_preprocessor(Xr_train)),("model",DecisionTreeClassifier(max_depth=5,class_weight="balanced",random_state=RANDOM_STATE))])}
for name,model in grade_models.items(): model.fit(Xg_train,yg_train); print("Trained grade model:",name)
for name,model in risk_models.items(): model.fit(Xr_train,yr_train); print("Trained risk model:",name)

## 5. SEMMA — Assess

Assessment must match the task. Regression uses MAE, RMSE, and R². Early-warning classification uses ROC-AUC, precision, recall, F1, accuracy, and a confusion matrix. The final recommendation considers both performance and whether the feature timing matches the intended use.

In [ ]:
grade_rows=[]
for name,model in grade_models.items():
    pred=model.predict(Xg_test)
    grade_rows.append({"model":name,"MAE":mean_absolute_error(yg_test,pred),"RMSE":mean_squared_error(yg_test,pred)**.5,"R2":r2_score(yg_test,pred)})
grade_results=pd.DataFrame(grade_rows).sort_values("RMSE"); display(grade_results.round(3))

In [ ]:
risk_rows=[]
for name,model in risk_models.items():
    prob=model.predict_proba(Xr_test)[:,1]; pred=(prob>=.5).astype(int)
    risk_rows.append({"model":name,"ROC_AUC":roc_auc_score(yr_test,prob),"accuracy":accuracy_score(yr_test,pred),"precision":precision_score(yr_test,pred,zero_division=0),"recall":recall_score(yr_test,pred,zero_division=0),"F1":f1_score(yr_test,pred,zero_division=0)})
risk_results=pd.DataFrame(risk_rows).sort_values("ROC_AUC",ascending=False); display(risk_results.round(3))
selected_risk="logistic_regression"; risk_prob=risk_models[selected_risk].predict_proba(Xr_test)[:,1]; risk_pred=(risk_prob>=.5).astype(int)
sns.heatmap(confusion_matrix(yr_test,risk_pred),annot=True,fmt="d",cmap="Blues",cbar=False); plt.title("Early-warning logistic regression confusion matrix"); plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.show()
print(classification_report(yr_test,risk_pred,target_names=["Not at risk","At risk"]))

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(13,5))
for name,model in risk_models.items():
    prob=model.predict_proba(Xr_test)[:,1]; fpr,tpr,_=roc_curve(yr_test,prob); axes[0].plot(fpr,tpr,label=f"{name} AUC={roc_auc_score(yr_test,prob):.3f}")
for name,model in grade_models.items(): axes[1].scatter(yg_test,model.predict(Xg_test),alpha=.55,label=name)
axes[0].plot([0,1],[0,1],"k--",alpha=.5); axes[0].set(title="Early-warning ROC",xlabel="False-positive rate",ylabel="True-positive rate"); axes[1].plot([0,20],[0,20],"k--",alpha=.5); axes[1].set(title="Actual vs predicted final grade",xlabel="Actual G3",ylabel="Predicted G3")
for ax in axes: ax.legend()
plt.tight_layout(); plt.show()

## SAS Enterprise Miner mapping

This Python notebook follows the same logical SEMMA stages used by SAS Enterprise Miner.

| SEMMA stage | Notebook implementation | SAS-style equivalent |
|---|---|---|
| Sample | Reproducible train/test partitions | Sample/partition nodes |
| Explore | Descriptive statistics and plots | StatExplore/graphical nodes |
| Modify | Imputation, encoding, scaling, timing-aware feature views | Transform/replacement/variable-selection nodes |
| Model | Regression, random forest, logistic regression, decision tree | Regression/tree/logistic nodes |
| Assess | Error metrics, ROC-AUC, confusion matrix, model comparison | Model comparison, fit statistics, lift/ROC assessment |

This is a Python implementation of the SEMMA logic. It should not be described as execution inside licensed SAS software unless you run it there and capture evidence.

## Deployment and artifacts

The grade model is a prediction aid, not a student label. The risk model should trigger supportive review such as tutoring or advising. Do not use it for punishment, denial of educational services, or decisions without human context.

In [ ]:
artifact_dir=Path("semma_artifacts"); artifact_dir.mkdir(exist_ok=True)
selected_grade="random_forest_regressor"
metrics={"dataset":str(data_path),"rows":int(len(df)),"columns":int(df.shape[1]),"grade_target":"G3","risk_definition":"G3 < 10","grade_feature_policy":"G1 and G2 included for end-of-term prediction","risk_feature_policy":"G1 and G2 excluded for early-warning demonstration","grade_results":grade_results.to_dict(orient="records"),"risk_results":risk_results.to_dict(orient="records"),"deployment_action":"Use predictions to prioritize supportive human review, not to make punitive decisions."}
(artifact_dir/"metrics.json").write_text(json.dumps(metrics,indent=2)); joblib.dump(grade_models[selected_grade],artifact_dir/"student_grade_model.joblib"); joblib.dump(risk_models[selected_risk],artifact_dir/"student_early_warning_model.joblib"); grade_results.to_csv(artifact_dir/"grade_model_comparison.csv",index=False); risk_results.to_csv(artifact_dir/"risk_model_comparison.csv",index=False)
new_student=X_early.iloc[[0]]
print("Example early-warning risk:",f"{risk_models[selected_risk].predict_proba(new_student)[:,1][0]:.2%}")
print("Saved:",[p.name for p in artifact_dir.iterdir()])

## Final SEMMA conclusion and verification checklist

The project samples the student records, explores academic and social variables, modifies the data with leakage-safe preprocessing, models both final-grade regression and early-warning classification, and assesses task-appropriate metrics.

Before submission, verify:

- The Kaggle student-performance dataset is cited in the report.
- `G3` is the final-grade target.
- `G1` and `G2` are included only for end-of-term prediction.
- The early-warning model excludes `G1` and `G2`.
- All transformations are fitted only on training data.
- Regression and classification metrics are not mixed.
- The SAS mapping is described honestly.
- Saved artifacts and screenshots are included in GitHub.